# Preprocessing and pipelines

Scalers, encoders, imputers and how to chain them so that nothing from the test set leaks
into training. Each tool on 3-5 numbers first.

**What's in here**
- `StandardScaler`: `mean_`, `scale_`, transform by hand; fit on train only
- `MinMaxScaler`, `RobustScaler`
- `Pipeline` and `named_steps`
- `ColumnTransformer` with numeric + text columns; `OneHotEncoder(handle_unknown="ignore")`
- `SimpleImputer`, `KBinsDiscretizer`, `FunctionTransformer`, `TransformedTargetRegressor`
- `set_output(transform="pandas")`
- why a pipeline keeps cross-validation honest
- classification: `predict_proba`, thresholds, confusion matrix, class imbalance
- the real data: customer type on meters, price spikes on hourly data

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", 30)
np.set_printoptions(precision=4, suppress=True)

## 1. StandardScaler by hand

Three training values. The scaler stores their mean and standard deviation.

In [2]:
from sklearn.preprocessing import StandardScaler

train_vals = pd.DataFrame({"x": [1.0, 2.0, 3.0]})
scaler = StandardScaler().fit(train_vals)
print("mean_  :", scaler.mean_)
print("scale_ :", scaler.scale_)        # population std, ddof=0

mean_  : [2.]
scale_ : [0.8165]


In [3]:
test_vals = pd.DataFrame({"x": [4.0, 5.0]})
print("by hand   :", ((test_vals["x"] - scaler.mean_[0]) / scaler.scale_[0]).values)
print("transform :", scaler.transform(test_vals).ravel())

by hand   : [2.4495 3.6742]
transform : [2.4495 3.6742]


4 is (4 − 2) / 0.816 = 2.45 standard deviations above the training mean. Test values can go
outside [−1, 1]; that is fine.

**Pitfall (leak):** fitting the scaler on train + test changes the numbers.

In [4]:
all_vals = pd.concat([train_vals, test_vals])
leaky = StandardScaler().fit(all_vals)
print("leaky mean_/scale_ :", leaky.mean_, leaky.scale_)
print("test scaled honest :", scaler.transform(test_vals).ravel())
print("test scaled leaky  :", leaky.transform(test_vals).ravel())

leaky mean_/scale_ : [3.] [1.4142]
test scaled honest : [2.4495 3.6742]
test scaled leaky  : [0.7071 1.4142]


Other scalers, same three values: `MinMaxScaler` maps train min→0 and max→1; `RobustScaler`
uses median and IQR so one outlier does not dominate.

In [5]:
from sklearn.preprocessing import MinMaxScaler, RobustScaler

v = pd.DataFrame({"x": [1.0, 2.0, 3.0, 100.0]})
pd.DataFrame({"x": v["x"],
              "standard": StandardScaler().fit_transform(v).ravel().round(2),
              "minmax": MinMaxScaler().fit_transform(v).ravel().round(2),
              "robust": RobustScaler().fit_transform(v).ravel().round(2)})

,x,standard,minmax,robust
0,1.0,-0.60,0.00,-0.06
1,2.0,-0.58,0.01,-0.02
2,3.0,-0.55,0.02,0.02
3,100.0,1.73,1.00,3.82


## 2. Pipeline

A pipeline runs the steps in order. `fit` fits every step on the training data only;
`predict` transforms then predicts. Five rows, `y = 2*x1 + 3`.

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

X = pd.DataFrame({"x1": [1, 2, 3, 4, 5]})
y = pd.Series([5, 7, 9, 11, 13])
pipe = Pipeline([("scale", StandardScaler()), ("ols", LinearRegression())])
pipe.fit(X, y)
print(pipe.named_steps)

{'scale': StandardScaler(), 'ols': LinearRegression()}


In [7]:
print("scaler mean_ :", pipe.named_steps["scale"].mean_)
print("ols coef_    :", pipe.named_steps["ols"].coef_, "(per 1 sd of x1, not per unit)")
print("predict [6]  :", pipe.predict(pd.DataFrame({"x1": [6]})))

scaler mean_ : [3.]
ols coef_    : [2.8284] (per 1 sd of x1, not per unit)
predict [6]  : [15.]


The coefficient is 2.83, not 2: it is per standard deviation of x1 (sd = 1.414), and
2 × 1.414 = 2.83. `make_pipeline` names the steps automatically.

In [8]:
from sklearn.pipeline import make_pipeline

mp = make_pipeline(StandardScaler(), LinearRegression()).fit(X, y)
list(mp.named_steps)

['standardscaler', 'linearregression']

## 3. ColumnTransformer: numbers and text together

A 4-row frame with one numeric column and one text column. Scale the number, one-hot the text.

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

mix = pd.DataFrame({"kwh": [10.0, 20.0, 30.0, 40.0],
                    "region": ["North", "South", "North", "West"]})
mix

,kwh,region
0,10.0,North
1,20.0,South
2,30.0,North
3,40.0,West


In [10]:
ct = ColumnTransformer([
    ("num", StandardScaler(), ["kwh"]),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), ["region"]),
])
arr = ct.fit_transform(mix)
print(ct.get_feature_names_out())
pd.DataFrame(arr, columns=ct.get_feature_names_out())

['num__kwh' 'cat__region_North' 'cat__region_South' 'cat__region_West']


,num__kwh,cat__region_North,cat__region_South,cat__region_West
0,-1.341641,1.0,0.0,0.0
1,-0.447214,0.0,1.0,0.0
2,0.447214,1.0,0.0,0.0
3,1.341641,0.0,0.0,1.0


Column names tell you which step made which column. `handle_unknown="ignore"` means a region
never seen in training becomes all zeros instead of an error.

In [11]:
new = pd.DataFrame({"kwh": [25.0], "region": ["East"]})
pd.DataFrame(ct.transform(new), columns=ct.get_feature_names_out())

,num__kwh,cat__region_North,cat__region_South,cat__region_West
0,0.0,0.0,0.0,0.0


`set_output(transform="pandas")` makes transformers return DataFrames with those names.

In [12]:
ct.set_output(transform="pandas")
ct.fit_transform(mix)

,num__kwh,cat__region_North,cat__region_South,cat__region_West
0,-1.341641,1.0,0.0,0.0
1,-0.447214,0.0,1.0,0.0
2,0.447214,1.0,0.0,0.0
3,1.341641,0.0,0.0,1.0


## 4. Imputer, binning, function transforms, target transform

`SimpleImputer` fills NaN with a statistic learned on training data.

In [13]:
from sklearn.impute import SimpleImputer

miss = pd.DataFrame({"x": [1.0, np.nan, 3.0]})
imp = SimpleImputer(strategy="mean").fit(miss)
print("statistics_:", imp.statistics_)
imp.transform(miss).ravel()

statistics_: [2.]


array([1., 2., 3.])

In [14]:
imp2 = SimpleImputer(strategy="mean", add_indicator=True).fit(miss)
pd.DataFrame(imp2.transform(miss), columns=imp2.get_feature_names_out())

,x,missingindicator_x
0,1.0,0.0
1,2.0,1.0
2,3.0,0.0


`add_indicator=True` adds a column marking which rows were missing, so the model can learn
"missing" itself.

`KBinsDiscretizer` turns numbers into bins; `FunctionTransformer` applies any function.

In [15]:
from sklearn.preprocessing import KBinsDiscretizer, FunctionTransformer

vals = pd.DataFrame({"x": [1.0, 2.0, 3.0, 10.0, 11.0]})
kb = KBinsDiscretizer(n_bins=2, encode="ordinal", strategy="quantile").fit(vals)
print("bin_edges_:", kb.bin_edges_[0])
print("bins      :", kb.transform(vals).ravel())

bin_edges_: [ 1.  3. 11.]
bins      : [0. 0. 1. 1. 1.]


In [16]:
log1p = FunctionTransformer(np.log1p)
pd.DataFrame({"x": [0, 9, 99], "log1p(x)": log1p.transform(pd.DataFrame({"x": [0, 9, 99]}))["x"].round(3)})

,x,log1p(x)
0,0,0.000
1,9,2.303
2,99,4.605


`TransformedTargetRegressor` fits on a transformed y (e.g. log) and predicts back on the
original scale for you.

In [17]:
from sklearn.compose import TransformedTargetRegressor

Xe = pd.DataFrame({"x1": [1, 2, 3, 4]})
ye = pd.Series([2.7, 7.4, 20.1, 54.6])          # roughly e**x1
ttr = TransformedTargetRegressor(regressor=LinearRegression(), func=np.log, inverse_func=np.exp)
ttr.fit(Xe, ye)
pd.DataFrame({"y": ye, "pred": ttr.predict(Xe).round(2)})

,y,pred
0,2.7,2.71
1,7.4,7.37
2,20.1,20.08
3,54.6,54.70


## 5. Why the pipeline keeps cross-validation honest

Inside `cross_val_score`, a pipeline refits the scaler on each training fold. A scaler fitted
once on all data before CV has already seen every test fold. The toy below is an exact line, so
every fold scores 1; the point is that the pipeline is the thing you pass to CV.

In [18]:
from sklearn.model_selection import cross_val_score

Xc = pd.DataFrame({"x1": [1, 2, 3, 4, 5, 6, 7, 8]})
yc = pd.Series([3, 5, 7, 9, 11, 13, 15, 17])
scores = cross_val_score(make_pipeline(StandardScaler(), LinearRegression()), Xc, yc, cv=4, scoring="r2")
print("fold R2:", scores.round(3))

fold R2: [1. 1. 1. 1.]


## 6. Classification: probabilities and thresholds

Six rows, one feature, label 1 when x is large. `LogisticRegression` returns a probability
per class; `predict` applies a 0.5 threshold.

In [19]:
from sklearn.linear_model import LogisticRegression

Xl = pd.DataFrame({"x": [1, 2, 3, 4, 5, 6]})
yl = pd.Series([0, 0, 0, 1, 1, 1])
clf = LogisticRegression().fit(Xl, yl)
proba = clf.predict_proba(Xl)
pd.DataFrame({"x": Xl["x"], "y": yl, "p(class 0)": proba[:, 0].round(3), "p(class 1)": proba[:, 1].round(3), "predict": clf.predict(Xl)})

,x,y,p(class 0),p(class 1),predict
0,1,0,0.943,0.057,0
1,2,0,0.843,0.157,0
2,3,0,0.637,0.363,0
3,4,1,0.363,0.637,1
4,5,1,0.157,0.843,1
5,6,1,0.057,0.943,1


**Pitfall:** `predict_proba(X)[:, 1]` is the probability of class 1. Column 0 is class 0.
Ranking customers by column 0 ranks the least likely first.

Change the threshold and the predictions change; the model did not.

In [20]:
p1 = proba[:, 1]
pd.DataFrame({"x": Xl["x"], "y": yl, "p1": p1.round(3), "pred@0.5": (p1 >= 0.5).astype(int), "pred@0.3": (p1 >= 0.3).astype(int)})

,x,y,p1,pred@0.5,pred@0.3
0,1,0,0.057,0,0
1,2,0,0.157,0,0
2,3,0,0.363,0,1
3,4,1,0.637,1,1
4,5,1,0.843,1,1
5,6,1,0.943,1,1


Confusion matrix: rows = true class, columns = predicted class.

In [21]:
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score

y_true = pd.Series([0, 0, 0, 1, 1, 1, 1, 1])
y_pred = pd.Series([0, 0, 1, 1, 1, 0, 0, 1])
cm = confusion_matrix(y_true, y_pred)
pd.DataFrame(cm, index=["true 0", "true 1"], columns=["pred 0", "pred 1"])

,pred 0,pred 1
true 0,2,1
true 1,2,3


In [22]:
print("precision class 1 = TP / (TP + FP) =", 3 / (3 + 1))
print("recall    class 1 = TP / (TP + FN) =", 3 / (3 + 2))
print(classification_report(y_true, y_pred))

precision class 1 = TP / (TP + FP) = 0.75
recall    class 1 = TP / (TP + FN) = 0.6
              precision    recall  f1-score   support

           0       0.50      0.67      0.57         3
           1       0.75      0.60      0.67         5

    accuracy                           0.62         8
   macro avg       0.62      0.63      0.62         8
weighted avg       0.66      0.62      0.63         8



AUC needs probabilities, not hard labels. Six rows where the ranking is good but one
positive scores below 0.5.

In [23]:
y6 = pd.Series([0, 0, 0, 1, 1, 1])
p6 = pd.Series([0.1, 0.4, 0.6, 0.45, 0.7, 0.9])
hard6 = (p6 >= 0.5).astype(int)
pd.DataFrame({"y": y6, "proba": p6, "hard": hard6})

,y,proba,hard
0,0,0.10,0
1,0,0.40,0
2,0,0.60,1
3,1,0.45,0
4,1,0.70,1
5,1,0.90,1


In [24]:
print("AUC with probabilities:", round(roc_auc_score(y6, p6), 3))
print("AUC with hard labels  :", round(roc_auc_score(y6, hard6), 3))

AUC with probabilities: 0.889


AUC with hard labels  : 0.667


With probabilities 8 of the 9 positive/negative pairs are ranked correctly (0.889). Hard labels
throw the ranking away and give 0.667.

## 7. Class imbalance

Ten rows, one positive. A model that always says 0 is 90% accurate and useless.

In [25]:
from sklearn.metrics import accuracy_score

y_imb = pd.Series([0, 0, 0, 0, 0, 0, 0, 0, 0, 1])
always_zero = pd.Series([0] * 10)
print("accuracy of always-0:", accuracy_score(y_imb, always_zero))
print("recall of always-0  :", 0, "(never finds the positive)")

accuracy of always-0: 0.9
recall of always-0  : 0 (never finds the positive)


`class_weight="balanced"` up-weights the rare class during fitting; then choose the threshold on
precision/recall, not on accuracy. Always print the majority-class accuracy as the baseline.

## 8. Real data: customer type from the meters table

In [26]:
meters = pd.read_csv("../data/meters.csv")
meters["region"] = meters["region"].str.title()
print(meters["customer_type"].value_counts())
meters.head(3)

customer_type
residential    266
sme             34
Name: count, dtype: int64


,meter_id,region,tariff,customer_type,annual_kwh_estimate,signup_date,has_solar
0,M100000,London,Fixed,sme,21622.0,2021-07-07,False
1,M100001,London,Fixed,residential,2286.0,2022-11-17,False
2,M100002,London,Fixed,residential,3665.0,2021-06-04,False


In [27]:
Xm = meters[["annual_kwh_estimate", "region", "tariff"]]
ym = (meters["customer_type"] == "sme").astype(int)
from sklearn.model_selection import train_test_split

Xm_tr, Xm_te, ym_tr, ym_te = train_test_split(Xm, ym, test_size=0.3, random_state=0, stratify=ym)

pre = ColumnTransformer([
    ("num", make_pipeline(SimpleImputer(strategy="median"), StandardScaler()), ["annual_kwh_estimate"]),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"), OneHotEncoder(handle_unknown="ignore")), ["region", "tariff"]),
])
clf_pipe = Pipeline([("pre", pre), ("logit", LogisticRegression(max_iter=1000))]).fit(Xm_tr, ym_tr)
p_te = clf_pipe.predict_proba(Xm_te)[:, 1]
print("majority-class accuracy:", round(1 - ym_te.mean(), 3))
print("accuracy               :", round(accuracy_score(ym_te, p_te >= 0.5), 3))
print("AUC                    :", round(roc_auc_score(ym_te, p_te), 3))

majority-class accuracy: 0.889
accuracy               : 1.0
AUC                    : 1.0


**Interview check:** an AUC of 1.0 is suspicious. Here it is real: SME meters have an
annual estimate around 25,000 kWh and residential around 3,200, so one feature separates
them perfectly. Say that out loud rather than celebrating.

In [28]:
meters.groupby("customer_type")["annual_kwh_estimate"].describe()[["min", "50%", "max"]].round(0)

,min,50%,max
customer_type,,,
residential,930.0,3143.0,5814.0
sme,12822.0,23382.0,36818.0


A harder, imbalanced problem on the hourly data: is this hour a price spike (top 5% of
training prices)? Chronological split, threshold fixed on train.

In [29]:
df = pd.read_csv("../data/hourly_power_clean.csv", parse_dates=["time"])
df["hour"] = df["time"].dt.hour
df["net_load"] = df["consumption_mwh"] - 900 * df["wind_ms"] - 12 * df["solar_wm2"]
split = int(len(df) * 0.8)
cut = df["price_eur_mwh"].iloc[:split].quantile(0.95)
df["spike"] = (df["price_eur_mwh"] > cut).astype(int)
feats = ["net_load", "temp_c", "wind_ms", "hour"]
tr, te = df.iloc[:split], df.iloc[split:]
print("spike rate train:", round(tr["spike"].mean(), 3), " test:", round(te["spike"].mean(), 3))

spike rate train: 0.05  test: 0.007


In [30]:
sp = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=1000)).fit(tr[feats], tr["spike"])
p_sp = sp.predict_proba(te[feats])[:, 1]
print("AUC:", round(roc_auc_score(te["spike"], p_sp), 3))
for thr in [0.5, 0.7, 0.9]:
    pred = (p_sp >= thr).astype(int)
    tp = ((pred == 1) & (te["spike"] == 1)).sum()
    fp = ((pred == 1) & (te["spike"] == 0)).sum()
    fn = ((pred == 0) & (te["spike"] == 1)).sum()
    print(f"threshold {thr}: precision {tp / max(tp + fp, 1):.2f}  recall {tp / max(tp + fn, 1):.2f}  flagged {pred.sum()}")

AUC: 0.704
threshold 0.5: precision 0.02  recall 0.54  flagged 854
threshold 0.7: precision 0.02  recall 0.38  flagged 503
threshold 0.9: precision 0.06  recall 0.35  flagged 156


The spike rate fell from 5% to below 2% in the test period (the 2022 gas spike is in
training), and precision is poor at every threshold: the regime changed. Recall falls and
precision rises as the threshold increases; pick it for the cost of a missed spike vs a false
alarm, on training data, and say out loud that the training regime is not the test regime.

## Persisting a pipeline

`joblib.dump(pipe, "model.joblib")` and `joblib.load(...)` save the whole fitted pipeline,
preprocessing included. (Not run here.)

## Quick reference

| Task | Call |
|---|---|
| scale, fit on train only | `sc = StandardScaler().fit(X_train)`; `sc.transform(X_test)` |
| chain steps | `Pipeline([("scale", StandardScaler()), ("ols", LinearRegression())])` |
| per-column preprocessing | `ColumnTransformer([("num", ..., num_cols), ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)])` |
| feature names after transform | `ct.get_feature_names_out()` |
| DataFrame output | `.set_output(transform="pandas")` |
| fill NaN | `SimpleImputer(strategy="median", add_indicator=True)` |
| log target | `TransformedTargetRegressor(regressor, func=np.log, inverse_func=np.exp)` |
| class probabilities | `clf.predict_proba(X)[:, 1]` |
| imbalance | `class_weight="balanced"`, then choose threshold |
| baseline accuracy | `1 - y.mean()` for a rare positive |